In [ ]:
!pip install streamlit transformers torch earthengine-api requests beautifulsoup4 googletrans==4.0.0-rc1 twilio folium streamlit-folium python-dotenv

  Using cached googletrans-4.0.0rc1-py3-none-any.whl
  Using cached httpx-0.13.3-py3-none-any.whl.metadata (25 kB)
  Using cached httpcore-0.9.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached h11-0.9.0-py2.py3-none-any.whl.metadata (8.1 kB)
Using cached httpx-0.13.3-py3-none-any.whl (55 kB)
Using cached httpcore-0.9.1-py3-none-any.whl (42 kB)
Using cached h11-0.9.0-py2.py3-none-any.whl (53 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.16.0
    Uninstalling h11-0.16.0:
      Successfully uninstalled h11-0.16.0
  Attempting uninstall: httpcore
    Found existing installation: httpcore 1.0.9
    Uninstalling httpcore-1.0.9:
      Successfully uninstalled httpcore-1.0.9
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fol

In [ ]:
!pip install streamlit transformers torch earthengine-api requests beautifulsoup4 googletrans==4.0.0-rc1 twilio folium streamlit-folium python-dotenv

In [ ]:
!pip install streamlit transformers torch earthengine-api requests beautifulsoup4 googletrans==4.0.0-rc1 twilio folium streamlit-folium python-dotenv

In [ ]:
!pip install streamlit transformers torch earthengine-api requests beautifulsoup4 googletrans==4.0.0-rc1 twilio folium streamlit-folium python-dotenv

In [ ]:
!pip install streamlit transformers torch earthengine-api requests beautifulsoup4 googletrans==4.0.0-rc1 twilio folium streamlit-folium python-dotenv

In [ ]:
!pip uninstall -y googletrans httpx httpcore
!pip install deep-translator httpx

Found existing installation: googletrans 4.0.0rc1
Uninstalling googletrans-4.0.0rc1:
  Successfully uninstalled googletrans-4.0.0rc1
Found existing installation: httpx 0.13.3
Uninstalling httpx-0.13.3:
  Successfully uninstalled httpx-0.13.3
Found existing installation: httpcore 0.9.1
Uninstalling httpcore-0.9.1:
  Successfully uninstalled httpcore-0.9.1
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.9.0
    Uninstalling h11-0.9.0:
      Successfully uninstalled h11-0.9.0


In [ ]:
!earthengine authenticate

E0000 00:00:1767864404.473029   22269 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767864404.503425   22269 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767864404.578540   22269 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767864404.579507   22269 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767864404.579546   22269 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767864404.579556   22269 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
import ee
ee.Initialize(project='banded-edge-425514-j1')
print(ee.String('Test').getInfo())  # Should print 'Test'


Test


In [ ]:
import os
from dotenv import load_dotenv
from transformers import pipeline

load_dotenv()  # Load secrets from .env if you have one; otherwise, use Colab secrets directly

HF_TOKEN = os.getenv('HF_TOKEN')  # Or use st.secrets['HF_TOKEN'] in Streamlit if integrated there

# Load the small climate LLM for risk classification
model_name = "KevSun/climate-attitude-LM"
classifier = pipeline("text-classification", model=model_name, token=HF_TOKEN)

# Test with sample input
sample_input = "Heavy rainfall in the Indus basin with rising river levels, potential for severe flooding and evacuations needed."
result = classifier(sample_input)
print(result)  # Example output: [{'label': 'risk', 'score': 0.95}] – interpret 'risk' as high flood risk

# Another test: Low risk scenario
low_risk_sample = "Normal weather conditions, no significant rainfall or flood threats."
low_result = classifier(low_risk_sample)
print(low_result)  # Likely [{'label': 'neutral' or 'opportunity', 'score': high}]

Device set to use cpu


[{'label': 'neutral', 'score': 0.9874022603034973}]
[{'label': 'neutral', 'score': 0.9822293519973755}]


In [ ]:
!pip install PyPDF2

import ee
import requests
from bs4 import BeautifulSoup
import PyPDF2
from io import BytesIO
import datetime
import os

# GEE already initialized in your setup

# Function for GEE rainfall (example for a village point, e.g., near Lahore)
def get_rainfall(lat=31.52, lon=74.35, days_back=1):
    try:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=days_back)
        point = ee.Geometry.Point(lon, lat)
        collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V06').filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        image = collection.mean()  # Average over period
        rainfall = image.select('precipitationCal').reduceRegion(ee.Reducer.mean(), point, 10000).get('precipitationCal')
        return rainfall.getInfo() or 0.0  # mm/hr, return 0 if None
    except Exception as e:
        print(f"GEE error: {e}")
        return 0.0

# Function for NDMA sitreps (scrape latest PDF link, extract text summary)
def get_ndma_sitrep():
    try:
        url = "https://www.ndma.gov.pk/sitreps"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        # Find latest sitrep link (assumes first <a> with "Daily Situation Report" or similar)
        for a in soup.find_all('a'):
            if 'Daily Situation Report' in a.text or 'Monsoon' in a.text:
                pdf_url = 'https://www.ndma.gov.pk' + a['href']
                pdf_response = requests.get(pdf_url)
                pdf_file = BytesIO(pdf_response.content)
                reader = PyPDF2.PdfReader(pdf_file)
                text = ""
                for page in reader.pages[:2]:  # Extract first 2 pages for summary
                    text += page.extract_text() + "\n"
                return text.strip() or "No recent flood alerts."
        return "No sitrep found."
    except Exception as e:
        print(f"NDMA scrape error: {e}")
        return "Error fetching NDMA data."

# Function for PMD forecasts (scrape main forecast text)
def get_pmd_forecast():
    try:
        url = "https://www.pmd.gov.pk/en/"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        # Look for forecast section; adjust selector based on site (often in <div class="content"> or <p> with keywords)
        forecast_div = soup.find('div', id='forecast') or soup.find('div', class_='content')
        if forecast_div:
            text = forecast_div.get_text(strip=True)
        else:
            # Fallback: find paragraphs with 'rain' or 'flood'
            text = ""
            for p in soup.find_all('p'):
                if 'rain' in p.text.lower() or 'flood' in p.text.lower():
                    text += p.text + "\n"
        return text.strip() or "No current forecast available."
    except Exception as e:
        print(f"PMD scrape error: {e}")
        return "Error fetching PMD data."

# Function for Thingspeak soil moisture (using public channel 2741111 for soil moisture)
def get_soil_moisture(channel_id=2741111):
    try:
        url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.json?results=1"
        response = requests.get(url)
        data = response.json()
        # Assume field1 is soil moisture percentage
        moisture = float(data['feeds'][0]['field1']) if data['feeds'] else 0.0
        return moisture
    except Exception as e:
        print(f"Thingspeak error: {e}")
        return 0.0

In [ ]:
import requests

def fetch_thingspeak(channel_id=9, field=1, last=1):
    url = f"https://api.thingspeak.com/channels/{channel_id}/fields/{field}.json?results={last}"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()
    feeds = data.get("feeds", [])
    if not feeds:
        return None
    # Get latest field value
    latest = feeds[-1].get(f"field{field}")
    return {"latest": latest, "feeds": feeds}

print(fetch_thingspeak(9, field=1, last=3))


{'latest': '0', 'feeds': [{'created_at': '2026-01-08T08:56:15Z', 'entry_id': 16790093, 'field1': '200'}, {'created_at': '2026-01-08T09:08:18Z', 'entry_id': 16790094, 'field1': '200'}, {'created_at': '2026-01-08T09:20:22Z', 'entry_id': 16790095, 'field1': '0'}]}


In [ ]:
from bs4 import BeautifulSoup
import requests
from datetime import datetime

def scrape_ndma_latest():
    base = "https://www.ndma.gov.pk/"
    try:
        r = requests.get(base, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        texts = []

        # Broad search for news-like links in common containers
        # Look for links in divs with class names like 'news', 'latest-updates', 'recent-posts'
        # Or more generally, look for links within the main content area.
        potential_news_containers = soup.find_all(lambda tag: tag.name == 'div' and \
                                                    (tag.get('class') and any(c in str(tag.get('class')) for c in ['news', 'latest', 'update', 'post']) or \
                                                     tag.get('id') and any(i in str(tag.get('id')) for i in ['news', 'latest'])))

        # If specific containers are found, prioritize them
        if potential_news_containers:
            for container in potential_news_containers:
                for a in container.find_all('a', limit=5):
                    text = a.get_text(strip=True)
                    if text and len(text) > 10: # Avoid very short links
                        texts.append(text)
        else:
            # Fallback: Just get some prominent links from the page that might be news
            for a in soup.find_all('a', limit=10):
                text = a.get_text(strip=True)
                if text and len(text) > 10 and not any(keyword in text.lower() for keyword in ['login', 'about', 'contact', 'home']): # Filter common non-news links
                    texts.append(text)

        return {"site": base, "snippets": texts[:3]} # Return top 3 snippets
    except Exception as e:
        print(f"NDMA scrape error: {e}")
        return {"error": str(e)}

def scrape_pmd_forecast():
    url = "https://www.pmd.gov.pk/en/"
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        snippets = []

        # First, try to find a specific forecast div as done in get_pmd_forecast
        forecast_div = soup.find('div', id='forecast') or soup.find('div', class_='content')

        if forecast_div:
            # Extract text from paragraphs within this div
            for p in forecast_div.find_all('p', limit=5):
                text = p.get_text(strip=True)
                if text and len(text) > 20: # Ensure it's substantial text
                    snippets.append(text)
        else:
            # Fallback: Search all paragraphs for weather-related keywords
            for p in soup.find_all('p', limit=10): # Limit paragraphs to check
                text = p.get_text(strip=True)
                if any(keyword in text.lower() for keyword in ['weather', 'forecast', 'rain', 'outlook', 'warning', 'advisory']) and len(text) > 20:
                    snippets.append(text)

        return {"snippets": snippets[:3]} # Return top 3 relevant snippets
    except Exception as e:
        print(f"PMD scrape error: {e}")
        return {"error": str(e)}


In [ ]:
!pip install deep-translator

from deep_translator import GoogleTranslator

# Function to process data and generate alerts
def generate_alert(lat, lon):
    rainfall = get_rainfall(lat, lon)
    soil_moisture = get_soil_moisture()
    ndma_text = get_ndma_sitrep()[:500]  # Truncate for input size
    pmd_forecast = get_pmd_forecast()[:500]

    # Combine into LLM input
    input_text = f"Rainfall: {rainfall} mm, Soil moisture: {soil_moisture}%, NDMA sitrep: {ndma_text}, PMD forecast: {pmd_forecast}. Assess climate risk."

    # Classify risk
    result = classifier(input_text)
    label = result[0]['label']
    score = result[0]['score']

    # Map to risk level and action plan
    if label == 'risk' and score > 0.7:
        risk_level = "High"
        action_plan = "Evacuate to higher ground, alert local authorities, stock essentials."
    elif label == 'neutral':
        risk_level = "Medium"
        action_plan = "Monitor closely, prepare emergency kit."
    else:
        risk_level = "Low"
        action_plan = "No immediate action, stay informed."

    # Multilingual output (e.g., Urdu)
    translator = GoogleTranslator(source='auto', target='ur')
    urdu_plan = translator.translate(action_plan)

    return risk_level, action_plan, urdu_plan

In [ ]:
%%writefile app.py
import streamlit as st
from twilio.rest import Client
import folium
from streamlit_folium import folium_static
import os

# Twilio setup
TWILIO_SID = os.getenv('TWILIO_SID')
TWILIO_TOKEN = os.getenv('TWILIO_TOKEN')
TWILIO_NUMBER = os.getenv('TWILIO_NUMBER')
client = Client(TWILIO_SID, TWILIO_TOKEN)

st.title("SkyGuard AI")

lat = st.number_input("Village Latitude", value=31.52)
lon = st.number_input("Village Longitude", value=74.35)
phone = st.text_input("Phone for SMS (e.g., +923001234567)")

if st.button("Generate Alert"):
    # Dummy logic — replace with your real function
    risk_level = "High"
    action_plan = "Evacuate low-lying areas immediately."
    urdu_plan = "فوری طور پر نشیبی علاقوں سے انخلا کریں۔"

    st.write(f"Risk Level: {risk_level}")
    st.write(f"Action Plan: {action_plan}")
    st.write(f"Urdu: {urdu_plan}")

    message = f"SkyGuard Alert: Risk {risk_level}. {action_plan} Urdu: {urdu_plan}"
    client.messages.create(from_=TWILIO_NUMBER, to=phone, body=message)
    st.success("SMS sent!")

    m = folium.Map(location=[lat, lon], zoom_start=10)
    color = 'red' if risk_level == 'High' else 'orange' if risk_level == 'Medium' else 'green'
    folium.Marker([lat, lon], popup=f"Risk: {risk_level}", icon=folium.Icon(color=color)).add_to(m)
    folium_static(m)

st.markdown("**Ethics Note:** This app uses AI for informational purposes only. Always verify with official sources like PMD/NDMA. Data privacy: No user data stored. Inclusive for rural areas via multilingual alerts.")


Overwriting app.py


In [ ]:
from google.colab import userdata
from twilio.rest import Client

TWILIO_SID = userdata.get('TWILIO_SID')
TWILIO_TOKEN = userdata.get('TWILIO_TOKEN')
TWILIO_NUMBER = userdata.get('TWILIO_NUMBER')

client = Client(TWILIO_SID, TWILIO_TOKEN)


In [1]:
import sys
# Ensure twilio is installed in the current session
!pip install -q twilio

from twilio.rest import Client
from google.colab import userdata

TWILIO_SID = userdata.get('TWILIO_SID')
TWILIO_TOKEN = userdata.get('TWILIO_TOKEN')
TWILIO_NUMBER = userdata.get('TWILIO_NUMBER')
RECIPIENT_PHONE = userdata.get('RECIPIENT_PHONE')

if TWILIO_SID and TWILIO_TOKEN:
    client = Client(TWILIO_SID, TWILIO_TOKEN)
    client.messages.create(
        from_=TWILIO_NUMBER,
        to=RECIPIENT_PHONE,
        body='SkyGuard AI test message'
    )
    print("Test SMS sent successfully!")
else:
    print("Twilio secrets are missing. Please add them in the Colab Secrets panel.")

ModuleNotFoundError: No module named 'twilio'

In [ ]:
%%writefile app.py
import streamlit as st
from twilio.rest import Client
import folium
from streamlit_folium import folium_static
import os

# Retrieve Twilio credentials from Streamlit secrets or OS environment
TWILIO_SID = os.getenv('TWILIO_SID') or st.secrets.get('TWILIO_SID')
TWILIO_TOKEN = os.getenv('TWILIO_TOKEN') or st.secrets.get('TWILIO_TOKEN')
TWILIO_NUMBER = os.getenv('TWILIO_NUMBER') or st.secrets.get('TWILIO_NUMBER')

if TWILIO_SID and TWILIO_TOKEN:
    client = Client(TWILIO_SID, TWILIO_TOKEN)
else:
    client = None

st.title("SkyGuard AI")

lat = st.number_input("Village Latitude", value=31.52)
lon = st.number_input("Village Longitude", value=74.35)
phone = st.text_input("Phone for SMS (e.g., +923001234567)")

if st.button("Generate Alert"):
    risk_level = "High"
    action_plan = "Evacuate low-lying areas immediately."
    urdu_plan = "فوری طور پر نشیبی علاقوں سے انخلا کریں۔"

    st.write(f"Risk Level: {risk_level}")
    st.write(f"Action Plan: {action_plan}")
    st.write(f"Urdu: {urdu_plan}")

    message = f"SkyGuard Alert: Risk {risk_level}. {action_plan} Urdu: {urdu_plan}"
    try:
        if client:
            client.messages.create(from_=TWILIO_NUMBER, to=phone, body=message)
            st.success("SMS sent!")
        else:
            st.warning("Twilio credentials not configured. SMS skipped.")
    except Exception as e:
        st.error(f"SMS failed: {e}")

    m = folium.Map(location=[lat, lon], zoom_start=10)
    color = 'red' if risk_level == 'High' else 'orange' if risk_level == 'Medium' else 'green'
    folium.Marker([lat, lon], popup=f"Risk: {risk_level}", icon=folium.Icon(color=color)).add_to(m)
    folium_static(m)

st.markdown("**Ethics Note:** This app uses AI for informational purposes only. Always verify with official sources like PMD/NDMA. Data privacy: No user data stored. Inclusive for rural areas via multilingual alerts.")

Overwriting app.py


In [ ]:
!rm -f /root/.config/ngrok/ngrok.yml


In [ ]:
from google.colab import userdata
import os

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
os.system(f"ngrok config add-authtoken {NGROK_AUTHTOKEN}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.138.62.203:8501

  Stopping...
🔗 Public URL: NgrokTunnel: "https://dacia-pseudoprofessorial-doreen.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import os
if os.path.exists('streamlit.log'):
    with open('streamlit.log', 'r') as f:
        print("\n--- Streamlit Logs ---")
        print(f.read())
        print("----------------------\n")
else:
    print("Streamlit log file not found. This might indicate Streamlit did not start or write logs as expected.")

In [ ]:
!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.138.62.203:8501

  Stopping...
🔗 Public URL: NgrokTunnel: "https://dacia-pseudoprofessorial-doreen.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.138.62.203:8501



In [ ]:
f!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)

In [ ]:
!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)

In [ ]:
!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)

In [ ]:
import time
import psutil
from pyngrok import ngrok
import os
import socket
from google.colab import userdata

# Ensure libraries are installed
!pip install -q twilio streamlit pyngrok

# Find and kill Streamlit processes
for proc in psutil.process_iter():
    try:
        if 'streamlit' in proc.name().lower() or any('streamlit run' in s for s in proc.cmdline()):
            print(f"Killing Streamlit process: {proc.pid} - {proc.name()})")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

# Find and kill ngrok processes
for proc in psutil.process_iter():
    try:
        if 'ngrok' in proc.name().lower():
            print(f"Killing ngrok process: {proc.pid} - {proc.name()})")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

print("Cleaned up previous Streamlit and ngrok processes.")

# Set your ngrok authtoken securely
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

# Confirm app.py exists
print("Checking for app.py:")
!ls -l /content/app.py

# Start Streamlit
!python -m streamlit run /content/app.py --server.port 8501 &

time.sleep(15)

def check_port(port, host='127.0.0.1'):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.connect((host, port))
        s.close()
        return True
    except (socket.error, ConnectionRefusedError):
        s.close()
        return False

port = 8501
print(f"\nChecking if Streamlit is listening on port {port}...")
max_attempts = 10
attempts = 0
while not check_port(port) and attempts < max_attempts:
    print(f"Port {port} not yet open, waiting... (Attempt {attempts + 1}/{max_attempts})")
    time.sleep(5)
    attempts += 1

if check_port(port):
    print(f"Streamlit is listening on port {port}. Proceeding with ngrok.")
    public_url = ngrok.connect(port)
    print("🔗 Public URL:", public_url)
else:
    print(f"Streamlit did not start on port {port} after {max_attempts} attempts.")

In [ ]:
import time
import psutil
from pyngrok import ngrok
import os
import socket
from google.colab import userdata

!pip install -q twilio streamlit pyngrok

# Find and kill processes
for proc in psutil.process_iter():
    try:
        if 'streamlit' in proc.name().lower() or any('streamlit run' in s for s in proc.cmdline()):
            proc.terminate()
        if 'ngrok' in proc.name().lower():
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, proc.ZombieProcess):
        pass

print("Cleaned up previous processes.")

# Set your ngrok authtoken securely
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

!nohup python -m streamlit run /content/app.py --server.port 8501 > streamlit.log 2>&1 &
time.sleep(10)

def check_port(port, host='127.0.0.1'):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.connect((host, port))
        s.close()
        return True
    except (socket.error, ConnectionRefusedError):
        s.close()
        return False

port = 8501
if check_port(port):
    public_url = ngrok.connect(port)
    print("🔗 Public URL:", public_url)
else:
    print("Streamlit did not start successfully. Check streamlit.log.")

In [ ]:
import time
import psutil
from pyngrok import ngrok
import os
import socket
from google.colab import userdata

# Ensure libraries are installed (redundant but robust)
!pip install twilio streamlit pyngrok

# Find and kill Streamlit processes
for proc in psutil.process_iter():
    try:
        if 'streamlit' in proc.name().lower() or any('streamlit run' in s for s in proc.cmdline()):
            print(f"Killing Streamlit process: {proc.pid} - {proc.name()}")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

# Find and kill ngrok processes
for proc in psutil.process_iter():
    try:
        if 'ngrok' in proc.name().lower():
            print(f"Killing ngrok process: {proc.pid} - {proc.name()}")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

print("Cleaned up previous Streamlit and ngrok processes.")

# Set your ngrok authtoken securely using Colab secrets
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

# Confirm app.py exists
print("Checking for app.py:")
!ls -l /content/app.py

# Start Streamlit in the background
!nohup python -m streamlit run /content/app.py --server.port 8501 > streamlit.log 2>&1 &

# Add a delay to ensure Streamlit has time to start
time.sleep(10)

# Check if Streamlit is actually listening on port 8501
def check_port(port, host='127.0.0.1'):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.connect((host, port))
        s.close()
        return True
    except (socket.error, ConnectionRefusedError):
        s.close()
        return False

port = 8501
print(f"\nChecking if Streamlit is listening on port {port}...")
max_attempts = 10
attempts = 0
while not check_port(port) and attempts < max_attempts:
    print(f"Port {port} not yet open, waiting... (Attempt {attempts + 1}/{max_attempts})")
    time.sleep(5)
    attempts += 1

if check_port(port):
    print(f"Streamlit is listening on port {port}. Proceeding with ngrok.")
    public_url = ngrok.connect(port)
    print("🔗 Public URL configured successfully.")
else:
    print(f"Streamlit did not start on port {port} after {max_attempts} attempts. Check streamlit.log for errors.")

if os.path.exists('streamlit.log'):
    with open('streamlit.log', 'r') as f:
        print("\n--- Streamlit Logs ---")
        print(f.read())
        print("----------------------\n")

In [ ]:
!curl https://loca.lt/mytunnelpassword


# Task
Integrate the `generate_alert` function and its dependencies into the `app.py` file. This will update the Streamlit application to dynamically generate flood risk alerts, action plans, and their Urdu translations using real-time data collection and AI classification, replacing the current hardcoded values.

## Integrate Alert Generation into Streamlit

### Subtask:
Modify the `app.py` file to import the `generate_alert` function and call it with the user-provided latitude and longitude. This will replace the hardcoded risk level and action plan with dynamic values generated by the data collection and LLM classification logic.


**Reasoning**:
The subtask requires modifying the `app.py` file to integrate the alert generation logic. This involves adding necessary imports, function definitions, and replacing hardcoded values with dynamic calls to the `generate_alert` function within the Streamlit app.



In [ ]:
%%writefile app.py
import streamlit as st
from twilio.rest import Client
import folium
from streamlit_folium import folium_static
import os
import ee
import requests
from bs4 import BeautifulSoup
import PyPDF2
from io import BytesIO
import datetime
from dotenv import load_dotenv
from transformers import pipeline
from deep_translator import GoogleTranslator

load_dotenv()

# Retrieve securely from the environment or system secrets
TWILIO_SID = os.getenv('TWILIO_SID') or st.secrets.get('TWILIO_SID')
TWILIO_TOKEN = os.getenv('TWILIO_TOKEN') or st.secrets.get('TWILIO_TOKEN')
TWILIO_NUMBER = os.getenv('TWILIO_NUMBER') or st.secrets.get('TWILIO_NUMBER')

if TWILIO_SID and TWILIO_TOKEN:
    client = Client(TWILIO_SID, TWILIO_TOKEN)
else:
    client = None

# Secure Earth Engine Project ID retrieval
EE_PROJECT = os.getenv('EE_PROJECT_ID') or st.secrets.get('EE_PROJECT_ID') or 'banded-edge-425514-j1'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception as e:
    st.error(f"Earth Engine failed to initialize: {e}")

HF_TOKEN = os.getenv('HF_TOKEN') or st.secrets.get('HF_TOKEN')
model_name = "KevSun/climate-attitude-LM"
if 'classifier' not in st.session_state:
    try:
        st.session_state.classifier = pipeline("text-classification", model=model_name, token=HF_TOKEN)
    except Exception as e:
        st.error(f"AI model failed to load: {e}")
classifier = st.session_state.classifier

# Helper functions
def get_rainfall(lat=31.52, lon=74.35, days_back=1):
    try:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=days_back)
        point = ee.Geometry.Point(lon, lat)
        collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V06').filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        image = collection.mean()
        if image is None: return 0.0
        rainfall = image.select('precipitationCal').reduceRegion(ee.Reducer.mean(), point, 10000).get('precipitationCal')
        return rainfall.getInfo() or 0.0
    except Exception as e:
        return 0.0

def get_ndma_sitrep():
    try:
        url = "https://www.ndma.gov.pk/sitreps"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for a in soup.find_all('a'):
            if 'Daily Situation Report' in a.text or 'Monsoon' in a.text:
                pdf_url = 'https://www.ndma.gov.pk' + a['href']
                pdf_response = requests.get(pdf_url, timeout=10)
                pdf_file = BytesIO(pdf_response.content)
                reader = PyPDF2.PdfReader(pdf_file)
                text = ""
                for page in reader.pages[:2]:
                    text += page.extract_text() + "\n"
                return text.strip() or "No recent flood alerts."
        return "No sitrep found."
    except Exception as e:
        return "Error fetching NDMA data."

def get_pmd_forecast():
    try:
        url = "https://www.pmd.gov.pk/en/"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        forecast_div = soup.find('div', id='forecast') or soup.find('div', class_='content')
        if forecast_div:
            text = forecast_div.get_text(strip=True)
        else:
            text = ""
            for p in soup.find_all('p'):
                if 'rain' in p.text.lower() or 'flood' in p.text.lower():
                    text += p.text + "\n"
        return text.strip() or "No current forecast available."
    except Exception as e:
        return "Error fetching PMD data."

def get_soil_moisture(channel_id=2741111):
    try:
        url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.json?results=1"
        response = requests.get(url, timeout=10)
        data = response.json()
        moisture = float(data['feeds'][0]['field1']) if data['feeds'] else 0.0
        return moisture
    except Exception as e:
        return 0.0

def generate_alert(lat, lon):
    rainfall = get_rainfall(lat, lon)
    soil_moisture = get_soil_moisture()
    ndma_text = get_ndma_sitrep()[:500]
    pmd_forecast = get_pmd_forecast()[:500]

    input_text = f"Rainfall: {rainfall} mm, Soil moisture: {soil_moisture}%, NDMA sitrep: {ndma_text}, PMD forecast: {pmd_forecast}. Assess climate risk."
    result = classifier(input_text)
    label = result[0]['label']
    score = result[0]['score']

    if label == 'risk' and score > 0.7:
        risk_level = "High"
        action_plan = "Evacuate to higher ground, alert local authorities, stock essentials."
    elif label == 'neutral':
        risk_level = "Medium"
        action_plan = "Monitor closely, prepare emergency kit."
    else:
        risk_level = "Low"
        action_plan = "No immediate action, stay informed."

    translator = GoogleTranslator(source='auto', target='ur')
    urdu_plan = translator.translate(action_plan)
    return risk_level, action_plan, urdu_plan

st.title("SkyGuard AI")

lat = st.number_input("Village Latitude", value=31.52)
lon = st.number_input("Village Longitude", value=74.35)
phone = st.text_input("Phone for SMS (e.g., +923001234567)")

if st.button("Generate Alert"):
    if 'classifier' not in st.session_state:
        st.error("AI model not loaded.")
    else:
        with st.spinner('Generating alert...'):
            risk_level, action_plan, urdu_plan = generate_alert(lat, lon)

        st.write(f"Risk Level: {risk_level}")
        st.write(f"Action Plan: {action_plan}")
        st.write(f"Urdu: {urdu_plan}")

        message = f

In [ ]:
%%writefile app.py
import streamlit as st
from twilio.rest import Client
import folium
from streamlit_folium import folium_static
import os
import ee
import requests
from bs4 import BeautifulSoup
import PyPDF2
from io import BytesIO
import datetime
from dotenv import load_dotenv
from transformers import pipeline
from deep_translator import GoogleTranslator

load_dotenv()

# Secure dynamic credential loading with robust fallback control
TWILIO_SID = os.getenv('TWILIO_SID') or st.secrets.get('TWILIO_SID')
TWILIO_TOKEN = os.getenv('TWILIO_TOKEN') or st.secrets.get('TWILIO_TOKEN')
TWILIO_NUMBER = os.getenv('TWILIO_NUMBER') or st.secrets.get('TWILIO_NUMBER')

if TWILIO_SID and TWILIO_TOKEN:
    try:
        client = Client(TWILIO_SID, TWILIO_TOKEN)
    except Exception:
        client = None
else:
    client = None

# Secure Earth Engine Project ID retrieval
EE_PROJECT = os.getenv('EE_PROJECT_ID') or st.secrets.get('EE_PROJECT_ID') or 'banded-edge-425514-j1'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception as e:
    st.error(f"Earth Engine failed to initialize: {e}")

HF_TOKEN = os.getenv('HF_TOKEN') or st.secrets.get('HF_TOKEN')
model_name = "KevSun/climate-attitude-LM"
if 'classifier' not in st.session_state:
    try:
        st.session_state.classifier = pipeline("text-classification", model=model_name, token=HF_TOKEN)
    except Exception as e:
        st.error(f"AI model failed to load: {e}")

# Helper functions
def get_rainfall(lat=31.52, lon=74.35, days_back=1):
    try:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=days_back)
        point = ee.Geometry.Point(lon, lat)
        collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V06').filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        image = collection.mean()
        if image is None: return 0.0
        rainfall = image.select('precipitationCal').reduceRegion(ee.Reducer.mean(), point, 10000).get('precipitationCal')
        return rainfall.getInfo() or 0.0
    except Exception as e:
        return 0.0

def get_ndma_sitrep():
    try:
        url = "https://www.ndma.gov.pk/sitreps"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for a in soup.find_all('a'):
            if 'Daily Situation Report' in a.text or 'Monsoon' in a.text:
                pdf_url = 'https://www.ndma.gov.pk' + a['href']
                pdf_response = requests.get(pdf_url, timeout=10)
                pdf_file = BytesIO(pdf_response.content)
                reader = PyPDF2.PdfReader(pdf_file)
                text = ""
                for page in reader.pages[:2]:
                    text += page.extract_text() + "\n"
                return text.strip() or "No recent flood alerts."
        return "No sitrep found."
    except Exception as e:
        return "Error fetching NDMA data."

def get_pmd_forecast():
    try:
        url = "https://www.pmd.gov.pk/en/"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        forecast_div = soup.find('div', id='forecast') or soup.find('div', class_='content')
        if forecast_div:
            text = forecast_div.get_text(strip=True)
        else:
            text = ""
            for p in soup.find_all('p'):
                if 'rain' in p.text.lower() or 'flood' in p.text.lower():
                    text += p.text + "\n"
        return text.strip() or "No current forecast available."
    except Exception as e:
        return "Error fetching PMD data."

def get_soil_moisture(channel_id=2741111):
    try:
        url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.json?results=1"
        response = requests.get(url, timeout=10)
        data = response.json()
        moisture = float(data['feeds'][0]['field1']) if data['feeds'] else 0.0
        return moisture
    except Exception as e:
        return 0.0

def generate_alert(lat, lon):
    rainfall = get_rainfall(lat, lon)
    soil_moisture = get_soil_moisture()
    ndma_text = get_ndma_sitrep()[:500]
    pmd_forecast = get_pmd_forecast()[:500]

    input_text = f"Rainfall: {rainfall} mm, Soil moisture: {soil_moisture}%, NDMA sitrep: {ndma_text}, PMD forecast: {pmd_forecast}. Assess climate risk."

    # Run AI risk classification prototype
    if 'classifier' in st.session_state:
        result = st.session_state.classifier(input_text)
        label = result[0]['label']
        score = result[0]['score']
    else:
        label = 'neutral'
        score = 0.5

    # Experimental Prototype logic (interpreting 'risk' classification accurately)
    if label.lower() == 'risk' and score > 0.6:
        risk_level = "High"
        action_plan = "Evacuate to higher ground, alert local authorities, stock essentials."
    elif label.lower() == 'neutral' or score > 0.4:
        risk_level = "Medium"
        action_plan = "Monitor closely, prepare emergency kit."
    else:
        risk_level = "Low"
        action_plan = "No immediate action, stay informed."

    translator = GoogleTranslator(source='auto', target='ur')
    try:
        urdu_plan = translator.translate(action_plan)
    except Exception:
        urdu_plan = "موجودہ حالات کی نگرانی کریں اور تیار رہیں۔" # Fallback local Urdu warning

    return risk_level, action_plan, urdu_plan

st.title("SkyGuard AI (Flood Risk Prototype)")

lat = st.number_input("Village Latitude", value=31.52)
lon = st.number_input("Village Longitude", value=74.35)
phone = st.text_input("Phone for SMS alerts (e.g., +923001234567)")

# Extra dynamic control switches to avoid automatic and anonymous SMS billing abuse
send_sms_check = st.checkbox("Send emergency SMS alert (Twilio verified users only)", value=False)

if st.button("Generate Alert"):
    with st.spinner('Generating experimental risk assessment...'):
        risk_level, action_plan, urdu_plan = generate_alert(lat, lon)

    st.subheader("Assessment Outcomes")
    st.write(f"**Risk Level:** {risk_level}")
    st.write(f"**Action Plan:** {action_plan}")
    st.write(f"**Urdu Translation:** {urdu_plan}")

    if send_sms_check and phone:
        message = f"SkyGuard Alert: Risk {risk_level}. Action: {action_plan}. Urdu: {urdu_plan}"
        try:
            if client:
                client.messages.create(from_=TWILIO_NUMBER, to=phone, body=message)
                st.success("SMS sent successfully!")
            else:
                st.warning("Twilio client not configured or disabled. SMS skipped.")
        except Exception as e:
            st.error(f"SMS failed: {e}")
    elif send_sms_check and not phone:
        st.warning("Please enter a valid phone number to dispatch the SMS.")

    m = folium.Map(location=[lat, lon], zoom_start=10)
    color = 'red' if risk_level == 'High' else 'orange' if risk_level == 'Medium' else 'green'
    folium.Marker([lat, lon], popup=f"Risk: {risk_level}", icon=folium.Icon(color=color)).add_to(m)
    folium_static(m)

st.markdown("**Ethics & Safety Note:** This app is an experimental flood-risk assessment prototype using AI models. Do not use as a sole safety metric. Verify with PMD/NDMA. SMS limits, phone verification, and strict credential isolation are applied for production safety.")


In [ ]:
%%writefile app.py
import streamlit as st
from twilio.rest import Client
import folium
from streamlit_folium import folium_static
import os
import ee
import requests
from bs4 import BeautifulSoup
import PyPDF2
from io import BytesIO
import datetime
from dotenv import load_dotenv
from transformers import pipeline
from deep_translator import GoogleTranslator

load_dotenv()

# Secure dynamic credential loading
TWILIO_SID = os.getenv('TWILIO_SID') or st.secrets.get('TWILIO_SID')
TWILIO_TOKEN = os.getenv('TWILIO_TOKEN') or st.secrets.get('TWILIO_TOKEN')
TWILIO_NUMBER = os.getenv('TWILIO_NUMBER') or st.secrets.get('TWILIO_NUMBER')

if TWILIO_SID and TWILIO_TOKEN:
    try:
        client = Client(TWILIO_SID, TWILIO_TOKEN)
    except Exception:
        client = None
else:
    client = None

# Secure Earth Engine Project ID retrieval
EE_PROJECT = os.getenv('EE_PROJECT_ID') or st.secrets.get('EE_PROJECT_ID') or 'banded-edge-425514-j1'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception as e:
    st.error(f"Earth Engine failed to initialize: {e}")

HF_TOKEN = os.getenv('HF_TOKEN') or st.secrets.get('HF_TOKEN')
model_name = "KevSun/climate-attitude-LM"
if 'classifier' not in st.session_state:
    try:
        st.session_state.classifier = pipeline("text-classification", model=model_name, token=HF_TOKEN)
    except Exception as e:
        st.error(f"AI model failed to load: {e}")

# Improved Helper functions
def get_rainfall(lat=31.52, lon=74.35, days_back=1):
    """Fetches rainfall data from the newer IMERG V07 Earth Engine dataset."""
    try:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=days_back)
        point = ee.Geometry.Point(lon, lat)
        # Updated to NASA/GPM_L3/IMERG_V07
        collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        image = collection.mean()
        if image is None:
            raise ValueError("No image collection found for these dates.")
        rainfall = image.select('precipitationCal').reduceRegion(ee.Reducer.mean(), point, 10000).get('precipitationCal')
        val = rainfall.getInfo()
        if val is None:
            raise ValueError("No pixel data at this coordinates point.")
        return val
    except Exception as e:
        # Raise or propagate so fallback / error notice displays rather than masking as 0.0 rain
        raise RuntimeError(f"IMERG V07 Fetch Error: {e}")

def get_ndma_sitrep():
    try:
        url = "https://www.ndma.gov.pk/sitreps"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for a in soup.find_all('a'):
            if 'Daily Situation Report' in a.text or 'Monsoon' in a.text:
                pdf_url = 'https://www.ndma.gov.pk' + a['href']
                pdf_response = requests.get(pdf_url, timeout=10)
                pdf_file = BytesIO(pdf_response.content)
                reader = PyPDF2.PdfReader(pdf_file)
                text = ""
                for page in reader.pages[:2]:
                    text += page.extract_text() + "\n"
                return text.strip() or "No recent flood alerts."
        return "Data unavailable (Sitrep Link Not Found)."
    except Exception as e:
        return f"Data unavailable (Scrape Error: {e})"

def get_pmd_forecast():
    try:
        url = "https://www.pmd.gov.pk/en/"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        forecast_div = soup.find('div', id='forecast') or soup.find('div', class_='content')
        if forecast_div:
            text = forecast_div.get_text(strip=True)
        else:
            text = ""
            for p in soup.find_all('p'):
                if 'rain' in p.text.lower() or 'flood' in p.text.lower():
                    text += p.text + "\n"
        return text.strip() or "No current forecast available."
    except Exception as e:
        return f"Data unavailable (Scrape Error: {e})"

def get_soil_moisture(channel_id=2741111, field=1):
    """Fetches live soil moisture sensor data with robust error propagation."""
    try:
        url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.json?results=1"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if data.get('feeds'):
            moisture = float(data['feeds'][0][f'field{field}'])
            return moisture
        raise ValueError("Feed collection is empty.")
    except Exception as e:
        raise RuntimeError(f"ThingSpeak Fetch Error: {e}")

def generate_alert(lat, lon):
    # 1. Gather Telemetry data with safe error fallbacks indicated
    try:
        rainfall = get_rainfall(lat, lon)
        rainfall_display = f"{rainfall:.2f} mm"
    except Exception as e:
        rainfall = None
        rainfall_display = f"Data unavailable ({e})"

    try:
        soil_moisture = get_soil_moisture(channel_id=2741111, field=1)
        soil_display = f"{soil_moisture:.1f}%"
    except Exception as e:
        soil_moisture = None
        soil_display = f"Data unavailable ({e})"

    ndma_text = get_ndma_sitrep()[:500]
    pmd_forecast = get_pmd_forecast()[:500]

    # 2. Build input context for model analysis
    input_text = f"Rainfall: {rainfall_display}, Soil moisture: {soil_display}, NDMA: {ndma_text}, PMD: {pmd_forecast}. Assess climate risk."

    if 'classifier' in st.session_state:
        result = st.session_state.classifier(input_text)
        label = result[0]['label']
        score = result[0]['score']
    else:
        label = 'neutral'
        score = 0.5

    # 3. Hybrid Risk-Assessment Decision Engine
    # Due to attitude bias in the KevSun model, we apply safe rule-based overrides on telemetry limits:
    is_override = False

    if (rainfall is not None and rainfall > 15.0) or (soil_moisture is not None and soil_moisture > 80.0):
        risk_level = "High"
        action_plan = "Critical: High rainfall or saturated soil detected. Evacuate low-lying areas, secure assets and move to high ground."
        is_override = True
    elif (rainfall is not None and rainfall > 5.0) or (soil_moisture is not None and soil_moisture > 60.0):
        risk_level = "Medium"
        action_plan = "Alert: Moderate threat. Prepare emergency kits, keep communication devices active, monitor stream channels."
        is_override = True
    else:
        # Fallback to model interpretation
        if label.lower() == 'risk' and score > 0.6:
            risk_level = "High"
            action_plan = "Evacuate to higher ground, alert local authorities, stock essentials."
        elif label.lower() == 'neutral' or score > 0.4:
            risk_level = "Medium"
            action_plan = "Monitor meteorological bulletins closely and verify escape routes."
        else:
            risk_level = "Low"
            action_plan = "No immediate action necessary. Continue monitoring public channels."

    translator = GoogleTranslator(source='auto', target='ur')
    try:
        urdu_plan = translator.translate(action_plan)
    except Exception:
        urdu_plan = "موجودہ حالات کی نگرانی کریں اور احتیاطی تدابیر اختیار کریں۔"

    metadata = {
        "rainfall": rainfall_display,
        "soil_moisture": soil_display,
        "override_applied": is_override,
        "raw_ai_label": label,
        "raw_ai_score": score
    }

    return risk_level, action_plan, urdu_plan, metadata

st.title("SkyGuard AI (Experimental Prototype)")

lat = st.number_input("Village Latitude", value=31.52)
lon = st.number_input("Village Longitude", value=74.35)
phone = st.text_input("Phone for SMS alerts (e.g., +923001234567)")

send_sms_check = st.checkbox("Send Emergency SMS Alert (Twilio Authorized Recipients Only)", value=False)

if st.button("Generate Alert"):
    with st.spinner('Generating flood risk assessment...'):
        risk_level, action_plan, urdu_plan, meta = generate_alert(lat, lon)

    st.subheader("Assessment Results")
    st.write(f"**Risk Classification Level:** {risk_level}")
    st.write(f"**Action Plan:** {action_plan}")
    st.write(f"**Urdu translation (اردو):** {urdu_plan}")

    with st.expander("View Collected Telemetry Metadata & AI Diagnostics"):
        st.json(meta)

    if send_sms_check and phone:
        message = f"SkyGuard Alert: Risk {risk_level}. Action: {action_plan} Urdu: {urdu_plan}"
        try:
            if client:
                client.messages.create(from_=TWILIO_NUMBER, to=phone, body=message)
                st.success("SMS Alert successfully dispatched.")
            else:
                st.warning("Twilio is not configured. SMS transmission skipped.")
        except Exception as e:
            st.error(f"SMS Dispatch Failed: {e}")
    elif send_sms_check and not phone:
        st.warning("Please provide a valid phone number to transmit alerts.")

    m = folium.Map(location=[lat, lon], zoom_start=10)
    color = 'red' if risk_level == 'High' else 'orange' if risk_level == 'Medium' else 'green'
    folium.Marker([lat, lon], popup=f"Risk: {risk_level}", icon=folium.Icon(color=color)).add_to(m)
    folium_static(m)

st.markdown("**Ethics & Safety Disclaimer:** This is an experimental flood-risk assessment prototype incorporating a hybrid decision engine. It is NOT certified for real-world emergency management. Verify conditions directly via PMD or NDMA channels.")


In [ ]:
%%writefile app.py
import streamlit as st
from twilio.rest import Client
import folium
from streamlit_folium import folium_static
import os
import ee
import requests
from bs4 import BeautifulSoup
import PyPDF2
from io import BytesIO
import datetime
from dotenv import load_dotenv
from transformers import pipeline
from deep_translator import GoogleTranslator

load_dotenv()

# Secure dynamic credential loading with fallback controls
TWILIO_SID = os.getenv('TWILIO_SID') or st.secrets.get('TWILIO_SID')
TWILIO_TOKEN = os.getenv('TWILIO_TOKEN') or st.secrets.get('TWILIO_TOKEN')
TWILIO_NUMBER = os.getenv('TWILIO_NUMBER') or st.secrets.get('TWILIO_NUMBER')
# Restrict Twilio recipient dynamically to authorized phone numbers
AUTHORIZED_RECIPIENT = os.getenv('RECIPIENT_PHONE') or st.secrets.get('RECIPIENT_PHONE')

if TWILIO_SID and TWILIO_TOKEN:
    try:
        client = Client(TWILIO_SID, TWILIO_TOKEN)
    except Exception:
        client = None
else:
    client = None

# Secure Earth Engine Project ID retrieval without hardcoding private projects
EE_PROJECT = os.getenv('EE_PROJECT_ID') or st.secrets.get('EE_PROJECT_ID')
if not EE_PROJECT:
    st.warning("EE_PROJECT_ID is not configured. Google Earth Engine might fail to initialize.")

try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception as e:
    st.error(f"Earth Engine failed to initialize: {e}")

HF_TOKEN = os.getenv('HF_TOKEN') or st.secrets.get('HF_TOKEN')
model_name = "KevSun/climate-attitude-LM"
if 'classifier' not in st.session_state:
    try:
        st.session_state.classifier = pipeline("text-classification", model=model_name, token=HF_TOKEN)
    except Exception as e:
        st.error(f"AI model failed to load: {e}")

# Helper functions
def get_rainfall(lat=31.52, lon=74.35, days_back=1):
    try:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=days_back)
        point = ee.Geometry.Point(lon, lat)
        collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        image = collection.mean()
        if image is None:
            raise ValueError("No image collection available.")
        rainfall = image.select('precipitationCal').reduceRegion(ee.Reducer.mean(), point, 10000).get('precipitationCal')
        val = rainfall.getInfo()
        if val is None:
            raise ValueError("No spatial data at coordinates.")
        return val
    except Exception as e:
        raise RuntimeError(f"IMERG V07 Fetch Error: {e}")

def get_ndma_sitrep():
    try:
        url = "https://www.ndma.gov.pk/sitreps"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for a in soup.find_all('a'):
            if 'Daily Situation Report' in a.text or 'Monsoon' in a.text:
                pdf_url = 'https://www.ndma.gov.pk' + a['href']
                pdf_response = requests.get(pdf_url, timeout=10)
                pdf_file = BytesIO(pdf_response.content)
                reader = PyPDF2.PdfReader(pdf_file)
                text = ""
                for page in reader.pages[:2]:
                    text += page.extract_text() + "\n"
                return text.strip() or "No recent flood alerts."
        return "Data unavailable (Sitrep link not found)."
    except Exception as e:
        return f"Data unavailable (Scrape Error: {e})"

def get_pmd_forecast():
    try:
        url = "https://www.pmd.gov.pk/en/"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        forecast_div = soup.find('div', id='forecast') or soup.find('div', class_='content')
        if forecast_div:
            text = forecast_div.get_text(strip=True)
        else:
            text = ""
            for p in soup.find_all('p'):
                if 'rain' in p.text.lower() or 'flood' in p.text.lower():
                    text += p.text + "\n"
        return text.strip() or "No current forecast available."
    except Exception as e:
        return f"Data unavailable (Scrape Error: {e})"

def get_soil_moisture(channel_id=2741111, field=1):
    try:
        url = f"https://api.thingspeak.com/channels/{channel_id}/feeds.json?results=1"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if data.get('feeds'):
            return float(data['feeds'][0][f'field{field}'])
        raise ValueError("Feed empty.")
    except Exception as e:
        raise RuntimeError(f"ThingSpeak Fetch Error: {e}")

def generate_alert(lat, lon):
    try:
        rainfall = get_rainfall(lat, lon)
        rainfall_display = f"{rainfall:.2f} mm/hr"
    except Exception as e:
        rainfall = None
        rainfall_display = f"Data unavailable ({e})"

    try:
        soil_moisture = get_soil_moisture()
        soil_display = f"{soil_moisture:.1f}%"
    except Exception as e:
        soil_moisture = None
        soil_display = f"Data unavailable ({e})"

    ndma_text = get_ndma_sitrep()[:500]
    pmd_forecast = get_pmd_forecast()[:500]

    # Strict Telemetry-Dependent Safe Overrides
    if rainfall is None or soil_moisture is None:
        risk_level = "Unknown"
        action_plan = "Safety Warning: Telemetry acquisition failed. Risk level cannot be calculated securely. Check local radio broadcasts immediately."
    elif rainfall > 15.0 or soil_moisture > 80.0:
        risk_level = "High"
        action_plan = "Critical: Heavy rainfall rate or saturated soil detected. Evacuate low-lying areas and move to high ground immediately."
    elif rainfall > 5.0 or soil_moisture > 60.0:
        risk_level = "Medium"
        action_plan = "Alert: Moderate threat levels detected. Prepare emergency resources and keep communication channels open."
    else:
        # Fallback to model assessment
        input_text = f"Rainfall: {rainfall_display}, Soil moisture: {soil_display}, NDMA: {ndma_text}, PMD: {pmd_forecast}. Assess risk."
        if 'classifier' in st.session_state:
            result = st.session_state.classifier(input_text)
            label = result[0]['label']
            score = result[0]['score']
        else:
            label = 'neutral'
            score = 0.5

        if label.lower() == 'risk' and score > 0.6:
            risk_level = "High"
            action_plan = "Evacuate to higher ground, alert local authorities, stock essentials."
        elif label.lower() == 'neutral' or score > 0.4:
            risk_level = "Medium"
            action_plan = "Monitor meteorological bulletins closely and verify escape routes."
        else:
            risk_level = "Low"
            action_plan = "No immediate action necessary. Continue monitoring public channels."

    translator = GoogleTranslator(source='auto', target='ur')
    try:
        urdu_plan = translator.translate(action_plan)
    except Exception:
        urdu_plan = "موجودہ حالات کی نگرانی کریں اور احتیاطی تدابیر اختیار کریں۔"

    meta = {
        "rainfall": rainfall_display,
        "soil_moisture": soil_display,
        "raw_ai_label": label if 'label' in locals() else "N/A",
        "raw_ai_score": score if 'score' in locals() else 0.0
    }
    return risk_level, action_plan, urdu_plan, meta

st.title("SkyGuard AI (Experimental Prototype)")

lat = st.number_input("Village Latitude", value=31.52)
lon = st.number_input("Village Longitude", value=74.35)
phone = st.text_input("Phone for SMS alerts (e.g., +923001234567)")

send_sms_check = st.checkbox("Send Emergency SMS Alert", value=False)

if st.button("Generate Alert"):
    with st.spinner('Generating flood risk assessment...'):
        risk_level, action_plan, urdu_plan, meta = generate_alert(lat, lon)

    st.subheader("Assessment Results")
    st.write(f"**Risk Classification Level:** {risk_level}")
    st.write(f"**Action Plan:** {action_plan}")
    st.write(f"**Urdu translation (اردو):** {urdu_plan}")

    with st.expander("View Collected Telemetry Metadata & AI Diagnostics"):
        st.json(meta)

    if send_sms_check and phone:
        # Ensure SMS alerts cannot be sent to arbitrary external accounts unless verified/authorized
        if AUTHORIZED_RECIPIENT and phone.strip() != AUTHORIZED_RECIPIENT.strip():
            st.error("SMS Dispatch Failed: Recipient phone number is not in the authorized Twilio list for this sandbox application.")
        else:
            message = f"SkyGuard Alert: Risk {risk_level}. Action: {action_plan} Urdu: {urdu_plan}"
            try:
                if client:
                    client.messages.create(from_=TWILIO_NUMBER, to=phone, body=message)
                    st.success("SMS Alert successfully dispatched.")
                else:
                    st.warning("Twilio is not configured. SMS transmission skipped.")
            except Exception as e:
                st.error(f"SMS Dispatch Failed: {e}")
    elif send_sms_check and not phone:
        st.warning("Please provide a valid phone number to transmit alerts.")

    m = folium.Map(location=[lat, lon], zoom_start=10)
    color = 'red' if risk_level == 'High' else 'orange' if risk_level == 'Medium' else 'blue' if risk_level == 'Unknown' else 'green'
    folium.Marker([lat, lon], popup=f"Risk: {risk_level}", icon=folium.Icon(color=color)).add_to(m)
    folium_static(m)

st.markdown("**Ethics & Safety Disclaimer:** This is an experimental flood-risk assessment prototype incorporating a hybrid decision engine. It is NOT certified for real-world emergency management. Verify conditions directly via PMD or NDMA channels.")

## Verify Data Collection and Test Streamlit App

### Subtask:
Run the updated `app.py` and interact with the Streamlit interface. This will allow us to verify that the `generate_alert` function is being called correctly, that all data collection (rainfall, soil moisture, NDMA, PMD) is functioning as expected, and that the SMS alerts are sent with the dynamically generated content.


**Reasoning**:
Before restarting the Streamlit application with the updated `app.py` file, it is essential to ensure that any previously running Streamlit processes and ngrok tunnels are terminated. This prevents conflicts and ensures the new version of the app is loaded correctly.



In [ ]:
import os
import psutil

# Find and kill Streamlit processes
for proc in psutil.process_iter():
    try:
        # Check if process name contains 'streamlit' or command line contains 'streamlit run'
        if 'streamlit' in proc.name().lower() or any('streamlit run' in s for s in proc.cmdline()):
            print(f"Killing Streamlit process: {proc.pid} - {proc.name()}")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

# Find and kill ngrok processes
for proc in psutil.process_iter():
    try:
        if 'ngrok' in proc.name().lower():
            print(f"Killing ngrok process: {proc.pid} - {proc.name()}")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

print("Cleaned up previous Streamlit and ngrok processes.")

**Reasoning**:
Restart the Streamlit application and create a new ngrok tunnel to access the updated `app.py`.



In [ ]:
!pip install pyngrok
import time

!streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(5)  # Wait for 5 seconds

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)

**Reasoning**:
The previous attempt to start Streamlit and ngrok failed because `streamlit` command was not found and ngrok authentication failed. I need to ensure `streamlit` is run using `python -m streamlit` and explicitly set the ngrok authtoken using `pyngrok` before connecting.



In [ ]:
import time
from pyngrok import ngrok
from google.colab import userdata

# Set your ngrok authtoken securely from Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

!python -m streamlit run app.py &

# Add a delay to ensure Streamlit has time to start
time.sleep(10)

public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)

In [ ]:
import time
import psutil
from pyngrok import ngrok
import os
import socket
from google.colab import userdata

# Ensure libraries are installed
!pip install twilio streamlit pyngrok

# Find and kill Streamlit processes
for proc in psutil.process_iter():
    try:
        if 'streamlit' in proc.name().lower() or any('streamlit run' in s for s in proc.cmdline()):
            print(f"Killing Streamlit process: {proc.pid} - {proc.name()}")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

# Find and kill ngrok processes
for proc in psutil.process_iter():
    try:
        if 'ngrok' in proc.name().lower():
            print(f"Killing ngrok process: {proc.pid} - {proc.name()}")
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

print("Cleaned up previous Streamlit and ngrok processes.")

# Set your ngrok authtoken securely using Colab secrets
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

# Confirm app.py exists
print("Checking for app.py:")
!ls -l /content/app.py

# Start Streamlit in the background
!nohup python -m streamlit run /content/app.py --server.port 8501 > streamlit.log 2>&1 &

time.sleep(5)

port = 8501
print(f"\nChecking if Streamlit is listening on port {port}...")
max_attempts = 15
attempts = 0
streamlit_ready = False
while not streamlit_ready and attempts < max_attempts:
    if os.path.exists('streamlit.log'):
        with open('streamlit.log', 'r') as f:
            log_content = f.read()
            if "You can now view your Streamlit app in your browser" in log_content:
                print("Streamlit successful startup message found in logs.")
                streamlit_ready = True
            elif "Port 8501 is already in use" in log_content:
                print(f"Error: Streamlit port {port} already in use.")
                break
    if not streamlit_ready:
        print(f"Streamlit not yet ready, waiting... (Attempt {attempts + 1}/{max_attempts})")
        time.sleep(5)
        attempts += 1

if streamlit_ready:
    print(f"Streamlit is listening on port {port}. Proceeding with ngrok.")
    public_url = ngrok.connect(port)
    print("🔗 Public URL:", public_url)
else:
    print(f"Streamlit did not start successfully after {max_attempts} attempts. Check streamlit.log.")

**Reasoning**:
The previous Streamlit startup failed with 'No module named streamlit'. Although `streamlit` was installed earlier, reinstalling it ensures it is available in the current Python environment and accessible via `python -m streamlit`. After successful installation, the Streamlit app will be launched and an ngrok tunnel will be established.



In [ ]:
import time
from pyngrok import ngrok
from google.colab import userdata

# Reinstall streamlit to ensure it's available
!pip install streamlit

# Set your ngrok authtoken securely using Colab secrets
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

# Start Streamlit in the background
!python -m streamlit run app.py &

time.sleep(15)

# Connect ngrok to the Streamlit port
public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.138.62.203:8502



# Task
Install the `twilio` library to resolve the `ModuleNotFoundError`. Then, restart the Streamlit application using `python -m streamlit` and establish a new ngrok tunnel to make it publicly accessible.

## Ensure Twilio is Installed and Restart Streamlit

### Subtask:
Install the `twilio` library, terminate existing Streamlit and ngrok processes, then restart the Streamlit application using `python -m streamlit` and establish a new ngrok tunnel.


**Reasoning**:
The previous Streamlit startup failed due to a missing 'twilio' module. Although it was installed earlier, it needs to be reinstalled to ensure its availability in the current environment for `app.py`.



In [ ]:
import time
import psutil
from pyngrok import ngrok
import os
from google.colab import userdata

!pip install -q twilio streamlit pyngrok

# Terminate existing setups
for proc in psutil.process_iter():
    try:
        if 'streamlit' in proc.name().lower() or any('streamlit run' in s for s in proc.cmdline()):
            proc.terminate()
        if 'ngrok' in proc.name().lower():
            proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

!nohup python -m streamlit run /content/app.py --server.port 8501 > streamlit.log 2>&1 &
time.sleep(15)

public_url = ngrok.connect(8501)
print("🔗 Public URL:", public_url)

In [ ]:
with open('streamlit.log', 'r') as f:
    print(f.read())

In [ ]:
with open('streamlit.log', 'r') as f:
    print(f.read())